# Silver Layer

Deduplicates and validates each Bronze table into its Silver equivalent via `MERGE INTO` Reads the full Bronze table each run rather than tracking what's
new since the last run.

In generation order: `customers`/`products` have no dependents among these, so
they run first and `order_items` assumes its parent `silver.orders` rows already
exist.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

CATALOG = "ecommerce_project"

## Entities with no injected issues: straight upsert

`products`, `order_items`, `payments`, and `events` don't have any of the five
injected data quality issues. Only `orders` and `customers` do.

In [0]:
def merge_into_silver(entity: str, primary_key: str) -> None:
    source_df = spark.table(f"{CATALOG}.bronze.{entity}")
    target_name = f"{CATALOG}.silver.{entity}"

    if not spark.catalog.tableExists(target_name):
        source_df.write.format("delta").saveAsTable(target_name)
        print(f"{entity}: created silver table, {source_df.count()} rows")
        return

    target = DeltaTable.forName(spark, target_name)
    (target.alias("target")
        .merge(source_df.alias("source"), f"target.{primary_key} = source.{primary_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"{entity}: merged, {spark.table(target_name).count()} rows total")

## customers: normalize the inconsistent country codes

In [0]:
COUNTRY_CODE_MAP = {
    "USA": "US", "United States": "US", "us": "US",
    "UK": "GB", "United Kingdom": "GB", "gb": "GB",
    "Germany": "DE", "de": "DE",
    "France": "FR", "fr": "FR",
    "Canada": "CA", "ca": "CA",
    "Japan": "JP", "jp": "JP",
}

def silver_customers() -> None:
    bronze_df = spark.table(f"{CATALOG}.bronze.customers")

    mapping_expr = F.create_map(*[F.lit(x) for pair in COUNTRY_CODE_MAP.items() for x in pair])
    cleaned = bronze_df.withColumn(
        "country_code",
        F.coalesce(mapping_expr[F.col("country_code")], F.col("country_code")),
    )

    target_name = f"{CATALOG}.silver.customers"
    if not spark.catalog.tableExists(target_name):
        cleaned.write.format("delta").saveAsTable(target_name)
        print(f"customers: created silver table, {cleaned.count()} rows")
        return

    target = DeltaTable.forName(spark, target_name)
    (target.alias("target")
        .merge(cleaned.alias("source"), "target.customer_id = source.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"customers: merged, {spark.table(target_name).count()} rows total")

## orders: dedup, quarantine what can't be trusted, merge the rest

Four issues here. Duplicates are deduplicated before the merge, since
`MERGE`'s join condition assumes at most one source row per key. The other
three (missing customer_id, malformed order_date, invalid total_amount) each
make a row impossible to trust as a real order, so they go to
`silver.orders_quarantine` instead of `silver.orders`. Which we can inspect later on.

In [0]:
def silver_orders() -> None:
    bronze_df = spark.table(f"{CATALOG}.bronze.orders")

    w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())
    deduped = (bronze_df
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn"))

    missing_customer = deduped.filter(F.col("customer_id").isNull())

    with_parsed_date = deduped.withColumn("parsed_date", F.to_date("order_date"))
    malformed_date = with_parsed_date.filter(
        F.col("customer_id").isNotNull() & F.col("parsed_date").isNull()
    )
    invalid_amount = with_parsed_date.filter(
        F.col("customer_id").isNotNull()
        & F.col("parsed_date").isNotNull()
        & (F.col("total_amount").isNull() | (F.col("total_amount") < 0))
    )
    clean = (with_parsed_date
        .filter(
            F.col("customer_id").isNotNull()
            & F.col("parsed_date").isNotNull()
            & F.col("total_amount").isNotNull()
            & (F.col("total_amount") >= 0)
        )
        .withColumn("order_date", F.col("parsed_date"))
        .drop("parsed_date"))

    quarantine = (
        missing_customer.withColumn("quarantine_reason", F.lit("missing_customer_id"))
        .unionByName(malformed_date.drop("parsed_date").withColumn("quarantine_reason", F.lit("malformed_order_date")))
        .unionByName(invalid_amount.drop("parsed_date").withColumn("quarantine_reason", F.lit("invalid_total_amount")))
    )

    quarantine_table = f"{CATALOG}.silver.orders_quarantine"
    write_mode = "append" if spark.catalog.tableExists(quarantine_table) else "overwrite"
    quarantine.write.format("delta").mode(write_mode).saveAsTable(quarantine_table)

    target_name = f"{CATALOG}.silver.orders"
    if not spark.catalog.tableExists(target_name):
        clean.write.format("delta").saveAsTable(target_name)
        print(f"orders: created silver table, {clean.count()} rows, quarantined {quarantine.count()}")
        return

    target = DeltaTable.forName(spark, target_name)
    (target.alias("target")
        .merge(clean.alias("source"), "target.order_id = source.order_id")
        .whenMatchedUpdate(
            condition="target.status != source.status",
            set={"status": "source.status"},
        )
        .whenNotMatchedInsertAll()
        .execute())
    print(f"orders: merged, {spark.table(target_name).count()} rows total, quarantined {quarantine.count()} this run")

## Call all functions defined

In [0]:
silver_customers()
merge_into_silver("products", "product_id")
silver_orders()
merge_into_silver("order_items", "order_item_id")
merge_into_silver("payments", "payment_id")
merge_into_silver("events", "event_id")
